# EdgeConnect TPU v5e: High-Resolution Islamic Pattern Restoration

## Optimized for Kaggle TPU v5e-8

**Configuration:**
- Resolution: 512×512
- Batch Size: 64
- Precision: bfloat16 (native TPU)
- Epochs: 90
- Target: Beat OpenCV (21.27 dB)

**Team Labaik**
- Talha Abid (L1F23BSAI0058)
- Abdullah Zafar (L1F23BSAI0054)
- Abdul Rehman (L1F23BSAI0050)
- Waqar Hussain (L1F23BSAI0078)

In [1]:
# TPU v5e Setup - PJRT Runtime Required
import os
os.environ['PJRT_DEVICE'] = 'TPU'  # Required for TPU v5e

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.nn.utils import spectral_norm

# TPU imports
import torch_xla
import torch_xla.core.xla_model as xm

print(f'PyTorch version: {torch.__version__}')
print(f'PyTorch XLA version: {torch_xla.__version__}')

# Get TPU device
DEVICE = torch_xla.device()
print(f'Using device: {DEVICE}')

# Test TPU
test = torch.ones(2, 2).to(DEVICE)
print(f'TPU test tensor: {test.device}')
print('TPU v5e is ready!')

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


PyTorch version: 2.8.0+cpu
PyTorch XLA version: 2.8.0


E0000 00:00:1766946125.899718      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


Using device: xla:0
TPU test tensor: xla:0
TPU v5e is ready!


In [2]:
# Load Dataset from Kaggle Input - DYNAMIC DETECTION
KAGGLE_INPUT = '/kaggle/input'

image_paths = []
print(f'Scanning {KAGGLE_INPUT} for datasets...')

for dataset_name in os.listdir(KAGGLE_INPUT):
    dataset_path = os.path.join(KAGGLE_INPUT, dataset_name)
    print(f'  Found dataset: {dataset_name}')
    
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                image_paths.append(os.path.join(root, file))

print(f'\nTotal images found: {len(image_paths)}')

if len(image_paths) == 0:
    print('\n⚠️ ERROR: No images found!')
    print('Make sure you have attached the islamic-patterns dataset.')
else:
    random.seed(42)
    random.shuffle(image_paths)
    
    NUM_SAMPLES = len(image_paths)
    n_train = int(0.8 * NUM_SAMPLES)
    n_val = int(0.1 * NUM_SAMPLES)
    
    train_paths = image_paths[:n_train]
    val_paths = image_paths[n_train:n_train+n_val]
    test_paths = image_paths[n_train+n_val:]
    
    print(f'Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}')

Scanning /kaggle/input for datasets...
  Found dataset: islamic-patterns-dataset

Total images found: 4176
Train: 3340, Val: 417, Test: 419


In [3]:
# Create Directories
DATA_DIR = '/kaggle/working/mughal_dataset'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'

for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

for split in ['train', 'val', 'test']:
    for subdir in ['images', 'masks', 'edges']:
        os.makedirs(f'{DATA_DIR}/{split}/{subdir}', exist_ok=True)

print('Directories created!')

Directories created!


In [4]:
# Mask Generation Functions
def generate_irregular_mask(height, width, min_strokes=1, max_strokes=4):
    '''Generate irregular free-form masks.'''
    mask = np.zeros((height, width), dtype=np.uint8)
    num_strokes = random.randint(min_strokes, max_strokes)
    
    for _ in range(num_strokes):
        start_x = random.randint(0, width - 1)
        start_y = random.randint(0, height - 1)
        
        for _ in range(random.randint(10, 30)):
            angle = random.uniform(0, 2 * np.pi)
            length = random.randint(10, 40)
            brush_w = random.randint(15, 50)
            
            end_x = int(start_x + length * np.cos(angle))
            end_y = int(start_y + length * np.sin(angle))
            end_x = np.clip(end_x, 0, width - 1)
            end_y = np.clip(end_y, 0, height - 1)
            
            cv2.line(mask, (start_x, start_y), (end_x, end_y), 255, brush_w)
            start_x, start_y = end_x, end_y
        
        if random.random() > 0.5:
            cx, cy = random.randint(0, width-1), random.randint(0, height-1)
            cv2.circle(mask, (cx, cy), random.randint(20, 60), 255, -1)
    
    return mask

def extract_canny_edges(image, low=50, high=150):
    '''Extract edges using Canny.'''
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)
    edges = cv2.Canny(blurred, low, high)
    return edges

print('Mask generation functions defined!')

Mask generation functions defined!


In [5]:
# Process Dataset - HIGH RESOLUTION 512x512
IMAGE_SIZE = 512  # High resolution for better pattern detail

def process_split(paths_list, split_name):
    success_count = 0
    for i, img_path in enumerate(tqdm(paths_list, desc=f'{split_name}')):
        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            img_np = np.array(img)
            
            mask = generate_irregular_mask(IMAGE_SIZE, IMAGE_SIZE)
            edges = extract_canny_edges(img_np)
            
            filename = f'{split_name}_{i:05d}.png'
            cv2.imwrite(f'{DATA_DIR}/{split_name}/images/{filename}', 
                       cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
            cv2.imwrite(f'{DATA_DIR}/{split_name}/masks/{filename}', mask)
            cv2.imwrite(f'{DATA_DIR}/{split_name}/edges/{filename}', edges)
            success_count += 1
        except Exception as e:
            continue
    return success_count

print(f'Processing images at {IMAGE_SIZE}x{IMAGE_SIZE}...')
train_count = process_split(train_paths, 'train')
val_count = process_split(val_paths, 'val')
test_count = process_split(test_paths, 'test')

print(f'\nDataset prepared!')
print(f'Train: {train_count}, Val: {val_count}, Test: {test_count}')

# Analyze Mask Coverage
def analyze_masks(split='test'):
    mask_dir = f'{DATA_DIR}/{split}/masks'
    mask_files = os.listdir(mask_dir)
    total_coverage = 0
    
    print(f'\nAnalyzing {split} mask coverage...')
    for f in tqdm(mask_files):
        mask = cv2.imread(os.path.join(mask_dir, f), 0)
        coverage = np.mean(mask > 0)
        total_coverage += coverage
        
    avg_coverage = (total_coverage / len(mask_files)) * 100
    print(f'Average Mask Coverage ({split}): {avg_coverage:.2f}%')
    return avg_coverage

coverage = analyze_masks('test')


Processing images at 512x512...


train:   0%|          | 0/3340 [00:00<?, ?it/s]

val:   0%|          | 0/417 [00:00<?, ?it/s]

test:   0%|          | 0/419 [00:00<?, ?it/s]


Dataset prepared!
Train: 3340, Val: 417, Test: 419

Analyzing test mask coverage...


  0%|          | 0/419 [00:00<?, ?it/s]

Average Mask Coverage (test): 11.81%


In [6]:
# EdgeConnect Models (optimized for TPU bfloat16)
class BaseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
    
    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

class EdgeGenerator(BaseNetwork):
    '''Stage 1: Predicts missing edges.'''
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels + 1, 64, 4, 2, 1), nn.LeakyReLU(0.2))
        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.enc3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.enc4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2))
        
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU())
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(512, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(256, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(128, out_channels, 4, 2, 1), nn.Sigmoid())
        
        self.init_weights()
    
    def forward(self, x, mask):
        inp = torch.cat([x, mask], dim=1)
        e1 = self.enc1(inp)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d4 = self.dec4(e4)
        d3 = self.dec3(torch.cat([d4, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))
        return d1

class InpaintGenerator(BaseNetwork):
    '''Stage 2: Fills texture guided by edges.'''
    def __init__(self, in_channels=4, out_channels=3):
        super().__init__()
        
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels + 1, 64, 4, 2, 1), nn.LeakyReLU(0.2))
        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.enc3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.enc4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2))
        
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU())
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(512, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU())
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(256, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(128, out_channels, 4, 2, 1), nn.Tanh())
        
        self.init_weights()
    
    def forward(self, x, edges, mask):
        inp = torch.cat([x, edges, mask], dim=1)
        e1 = self.enc1(inp)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d4 = self.dec4(e4)
        d3 = self.dec3(torch.cat([d4, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))
        return d1

class Discriminator(BaseNetwork):
    '''PatchGAN discriminator with Spectral Normalization.'''
    def __init__(self, in_channels=3):
        super().__init__()
        self.model = nn.Sequential(
            spectral_norm(nn.Conv2d(in_channels, 64, 4, 2, 1)),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv2d(64, 128, 4, 2, 1)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv2d(128, 256, 4, 2, 1)),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv2d(256, 512, 4, 1, 1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
            spectral_norm(nn.Conv2d(512, 1, 4, 1, 1))
        )
        self.init_weights()
    
    def forward(self, x):
        return self.model(x)

print('Models defined!')
print(f'EdgeGenerator params: {sum(p.numel() for p in EdgeGenerator().parameters()):,}')
print(f'InpaintGenerator params: {sum(p.numel() for p in InpaintGenerator().parameters()):,}')

Models defined!
EdgeGenerator params: 6,170,625
InpaintGenerator params: 6,175,747


In [7]:
# Dataset Class with Data Augmentation
class InpaintDataset(Dataset):
    def __init__(self, data_dir, split='train', augment=True):
        self.img_dir = os.path.join(data_dir, split, 'images')
        self.mask_dir = os.path.join(data_dir, split, 'masks')
        self.edge_dir = os.path.join(data_dir, split, 'edges')
        self.augment = augment and split == 'train'
        
        self.files = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.png')])
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filename = self.files[idx]
        
        img = cv2.imread(os.path.join(self.img_dir, filename))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        
        mask = cv2.imread(os.path.join(self.mask_dir, filename), 0)
        mask = Image.fromarray(mask)
        
        edges = cv2.imread(os.path.join(self.edge_dir, filename), 0)
        edges = Image.fromarray(edges)
        
        # Data Augmentation
        if self.augment:
            k = random.randint(0, 3)
            if k > 0:
                img = img.rotate(k * 90)
                mask = mask.rotate(k * 90)
                edges = edges.rotate(k * 90)
            
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
                edges = edges.transpose(Image.FLIP_LEFT_RIGHT)
        
        img = np.array(img)
        mask = np.array(mask)
        edges = np.array(edges)
        
        img_tensor = self.transform(img)
        mask_tensor = torch.from_numpy(mask).float().unsqueeze(0) / 255.0
        edges_tensor = torch.from_numpy(edges).float().unsqueeze(0) / 255.0
        
        masked_img = img_tensor * (1 - mask_tensor)
        
        return {
            'image': img_tensor,
            'masked_image': masked_img,
            'mask': mask_tensor,
            'edges': edges_tensor
        }

train_dataset = InpaintDataset(DATA_DIR, 'train', augment=True)
val_dataset = InpaintDataset(DATA_DIR, 'val', augment=False)
test_dataset = InpaintDataset(DATA_DIR, 'test', augment=False)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

Train: 3340, Val: 417, Test: 419


In [8]:
# TPU Training Function with bfloat16
def train_model_tpu(edge_gen, inpaint_gen, disc, train_loader, val_loader=None, epochs=90, lr=0.0002):
    '''Train on TPU with bfloat16.'''
    
    # Move to TPU and convert to bfloat16
    edge_gen = edge_gen.to(DEVICE).to(torch.bfloat16)
    inpaint_gen = inpaint_gen.to(DEVICE).to(torch.bfloat16)
    disc = disc.to(DEVICE).to(torch.bfloat16)
    
    opt_edge = optim.Adam(edge_gen.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_inpaint = optim.Adam(inpaint_gen.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_disc = optim.Adam(disc.parameters(), lr=lr * 0.5, betas=(0.0, 0.9))
    
    scheduler_edge = optim.lr_scheduler.CosineAnnealingLR(opt_edge, T_max=epochs, eta_min=1e-6)
    scheduler_inpaint = optim.lr_scheduler.CosineAnnealingLR(opt_inpaint, T_max=epochs, eta_min=1e-6)
    scheduler_disc = optim.lr_scheduler.CosineAnnealingLR(opt_disc, T_max=epochs, eta_min=1e-7)
    
    l1_loss = nn.L1Loss()
    adv_loss = nn.BCEWithLogitsLoss()
    
    history = {'edge_loss': [], 'inpaint_loss': [], 'disc_loss': [], 'psnr': []}
    best_psnr = 0
    
    print('\n' + '='*60)
    print('TPU v5e TRAINING CONFIGURATION')
    print('='*60)
    print(f'Device: TPU v5e (bfloat16)')
    print(f'Resolution: 512x512')
    print(f'Batch Size: {train_loader.batch_size}')
    print(f'Epochs: {epochs}')
    print(f'Label Smoothing: 0.9/0.1')
    print(f'Spectral Norm: Enabled')
    print(f'Data Augmentation: Enabled')
    print('='*60 + '\n')
    
    for epoch in range(epochs):
        edge_gen.train()
        inpaint_gen.train()
        disc.train()
        
        epoch_edge_loss = 0
        epoch_inpaint_loss = 0
        epoch_disc_loss = 0
        num_batches = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch in pbar:
            # Move to TPU and convert to bfloat16
            images = batch['image'].to(DEVICE).to(torch.bfloat16)
            masked_images = batch['masked_image'].to(DEVICE).to(torch.bfloat16)
            masks = batch['mask'].to(DEVICE).to(torch.bfloat16)
            edges = batch['edges'].to(DEVICE).to(torch.bfloat16)
            
            gray = images.mean(dim=1, keepdim=True)
            masked_gray = gray * (1 - masks)
            
            # Stage 1: Edge Generator
            opt_edge.zero_grad()
            pred_edges = edge_gen(masked_gray, masks)
            edge_l1 = l1_loss(pred_edges, edges)
            edge_l1.backward()
            torch.nn.utils.clip_grad_norm_(edge_gen.parameters(), 1.0)
            xm.optimizer_step(opt_edge)
            
            # Stage 2: Inpaint Generator
            opt_inpaint.zero_grad()
            pred_edges_detach = pred_edges.detach()
            pred_images = inpaint_gen(masked_images, pred_edges_detach, masks)
            
            recon_loss = l1_loss(pred_images, images)
            fake_pred = disc(pred_images)
            gen_adv_loss = adv_loss(fake_pred, torch.ones_like(fake_pred) * 0.9)
            total_inpaint_loss = recon_loss + 0.1 * gen_adv_loss
            
            total_inpaint_loss.backward()
            torch.nn.utils.clip_grad_norm_(inpaint_gen.parameters(), 1.0)
            xm.optimizer_step(opt_inpaint)
            
            # Discriminator
            opt_disc.zero_grad()
            real_pred = disc(images)
            fake_pred = disc(pred_images.detach())
            
            disc_real_loss = adv_loss(real_pred, torch.ones_like(real_pred) * 0.9)
            disc_fake_loss = adv_loss(fake_pred, torch.ones_like(fake_pred) * 0.1)
            disc_loss = (disc_real_loss + disc_fake_loss) / 2
            
            disc_loss.backward()
            torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
            xm.optimizer_step(opt_disc)
            
            # TPU sync point
            torch_xla.sync()
            
            epoch_edge_loss += edge_l1.item()
            epoch_inpaint_loss += total_inpaint_loss.item()
            epoch_disc_loss += disc_loss.item()
            num_batches += 1
            
            pbar.set_postfix({
                'edge': f'{edge_l1.item():.4f}',
                'inpaint': f'{recon_loss.item():.4f}',
                'disc': f'{disc_loss.item():.4f}'
            })
        
        avg_edge = epoch_edge_loss / max(num_batches, 1)
        avg_inpaint = epoch_inpaint_loss / max(num_batches, 1)
        avg_disc = epoch_disc_loss / max(num_batches, 1)
        
        # Validation PSNR
        if val_loader is not None:
            inpaint_gen.eval()
            edge_gen.eval()
            psnr_total = 0.0
            num_valid = 0
            with torch.no_grad():
                for batch in val_loader:
                    images = batch['image'].to(DEVICE).to(torch.float32)
                    masked_images = batch['masked_image'].to(DEVICE).to(torch.bfloat16)
                    masks = batch['mask'].to(DEVICE).to(torch.bfloat16)
                    
                    pred_edges = edge_gen(images.mean(dim=1, keepdim=True).to(torch.bfloat16) * (1-masks), masks)
                    pred_images = inpaint_gen(masked_images, pred_edges, masks).float()
                    
                    mse = torch.mean((pred_images - images) ** 2)
                    if mse > 0:
                        psnr = 10 * torch.log10(4 / (mse + 1e-8))
                        psnr_total += psnr.item()
                        num_valid += 1
                    torch_xla.sync()
            
            avg_psnr = psnr_total / max(num_valid, 1)
            history['psnr'].append(avg_psnr)
            print(f'Epoch {epoch+1}: Edge={avg_edge:.4f}, Inpaint={avg_inpaint:.4f}, Disc={avg_disc:.4f}, PSNR={avg_psnr:.2f} dB')
            
            if avg_psnr > best_psnr:
                best_psnr = avg_psnr
                xm.save({
                    'edge_gen': edge_gen.state_dict(),
                    'inpaint_gen': inpaint_gen.state_dict(),
                    'disc': disc.state_dict(),
                    'epoch': epoch + 1,
                    'best_psnr': best_psnr,
                }, f'{CHECKPOINT_DIR}/best_model.pth')
                print(f' -> New best model saved! PSNR: {best_psnr:.2f} dB')
        else:
            print(f'Epoch {epoch+1}: Edge={avg_edge:.4f}, Inpaint={avg_inpaint:.4f}, Disc={avg_disc:.4f}')
        
        scheduler_edge.step()
        scheduler_inpaint.step()
        scheduler_disc.step()
        
        history['edge_loss'].append(avg_edge)
        history['inpaint_loss'].append(avg_inpaint)
        history['disc_loss'].append(avg_disc)
    
    return history

print('TPU Training function defined!')

TPU Training function defined!


In [9]:
# Run TPU Training
edge_gen = EdgeGenerator(in_channels=1)
inpaint_gen = InpaintGenerator()
disc = Discriminator()

# Batch size for TPU v5e (64 for stability)
BATCH_SIZE = 64
EPOCHS = 90

# TPU DataLoader - no pin_memory, num_workers=0 for stability
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=0, drop_last=False)

print(f'\n' + '='*60)
print('EDGECONNECT TPU v5e TRAINING')
print('='*60)
print(f'Resolution: 512x512')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Train samples: {len(train_dataset)}')
print(f'Val samples: {len(val_dataset)}')
print(f'Device: TPU v5e (bfloat16)')
print('='*60 + '\n')

history = train_model_tpu(edge_gen, inpaint_gen, disc, train_loader, val_loader, epochs=EPOCHS)


EDGECONNECT TPU v5e TRAINING
Resolution: 512x512
Batch size: 64
Epochs: 90
Train samples: 3340
Val samples: 417
Device: TPU v5e (bfloat16)


TPU v5e TRAINING CONFIGURATION
Device: TPU v5e (bfloat16)
Resolution: 512x512
Batch Size: 64
Epochs: 90
Label Smoothing: 0.9/0.1
Spectral Norm: Enabled
Data Augmentation: Enabled



Epoch 1/90:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 1: Edge=0.2918, Inpaint=0.3028, Disc=0.7362, PSNR=17.66 dB
 -> New best model saved! PSNR: 17.66 dB


Epoch 2/90:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 2: Edge=0.2300, Inpaint=0.2152, Disc=0.7091, PSNR=19.61 dB
 -> New best model saved! PSNR: 19.61 dB


...

Epoch 88: Edge=0.0602, Inpaint=0.1277, Disc=0.6962, PSNR=22.12 dB


Epoch 89/90:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 89: Edge=0.0602, Inpaint=0.1275, Disc=0.6967, PSNR=22.03 dB


Epoch 90/90:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 90: Edge=0.0602, Inpaint=0.1276, Disc=0.6959, PSNR=22.04 dB


In [10]:
# Final Summary and Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['psnr'])
axes[0].axhline(y=21.27, color='r', linestyle='--', label='OpenCV Baseline')
axes[0].set_title('Validation PSNR')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('PSNR (dB)')
axes[0].legend()

axes[1].plot(history['disc_loss'], label='Disc')
axes[1].set_title('Discriminator Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_history.png', dpi=150)
plt.show()

print('\n' + '='*60)
print('EXPERIMENT COMPLETE')
print('='*60)
print(f'Best PSNR: {max(history["psnr"]):.2f} dB')
print(f'OpenCV Baseline: 21.27 dB')
if max(history['psnr']) > 21.27:
    print('🎉 SUCCESS: Beat the baseline!')
else:
    print(f'Gap to baseline: {21.27 - max(history["psnr"]):.2f} dB')


EXPERIMENT COMPLETE
Best PSNR: 22.59 dB
OpenCV Baseline: 21.27 dB
🎉 SUCCESS: Beat the baseline!


In [11]:
# Evaluation on Test Set
!pip install -q scikit-image

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

edge_gen.eval()
inpaint_gen.eval()

psnr_scores = []
ssim_scores = []

print('Evaluating on test set...')
with torch.no_grad():
    for batch in tqdm(test_loader):
        images = batch['image'].to(DEVICE).to(torch.float32)
        masked_images = batch['masked_image'].to(DEVICE).to(torch.bfloat16)
        masks = batch['mask'].to(DEVICE).to(torch.bfloat16)
        
        gray = images.mean(dim=1, keepdim=True).to(torch.bfloat16)
        masked_gray = gray * (1 - masks)
        
        pred_edges = edge_gen(masked_gray, masks)
        pred_images = inpaint_gen(masked_images, pred_edges, masks).float()
        torch_xla.sync()
        
        images_np = ((images + 1) / 2 * 255).cpu().numpy().transpose(0, 2, 3, 1).astype(np.uint8)
        pred_np = ((pred_images + 1) / 2 * 255).cpu().numpy().transpose(0, 2, 3, 1).astype(np.uint8)
        
        for i in range(images_np.shape[0]):
            psnr_val = psnr(images_np[i], pred_np[i], data_range=255)
            ssim_val = ssim(cv2.cvtColor(images_np[i], cv2.COLOR_RGB2GRAY),
                           cv2.cvtColor(pred_np[i], cv2.COLOR_RGB2GRAY), data_range=255)
            psnr_scores.append(psnr_val)
            ssim_scores.append(ssim_val)

print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)
print(f'PSNR: {np.mean(psnr_scores):.2f} ± {np.std(psnr_scores):.2f} dB')
print(f'SSIM: {np.mean(ssim_scores):.4f} ± {np.std(ssim_scores):.4f}')
print('='*60)


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Evaluating on test set...


  0%|          | 0/53 [00:00<?, ?it/s]


FINAL RESULTS
PSNR: 23.36 ± 3.86 dB
SSIM: 0.8863 ± 0.0518


In [12]:
# Visualize Results
def visualize_results(edge_model, inpaint_model, dataset, num_samples=5):
    edge_model.eval()
    inpaint_model.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    indices = random.sample(range(len(dataset)), num_samples)
    
    for i, idx in enumerate(indices):
        sample = dataset[idx]
        # Handle TPU/CPU tensors
        image = sample['image'].unsqueeze(0).to(DEVICE).to(torch.float32)
        masked_image = sample['masked_image'].unsqueeze(0).to(DEVICE).to(torch.bfloat16)
        mask = sample['mask'].unsqueeze(0).to(DEVICE).to(torch.bfloat16)
        
        gray = image.mean(dim=1, keepdim=True).to(torch.bfloat16)
        masked_gray = gray * (1 - mask)
        
        with torch.no_grad():
            pred_edges = edge_model(masked_gray, mask)
            pred_image = inpaint_model(masked_image, pred_edges, mask).float()
            torch_xla.sync()
        
        def to_display(t):
            return ((t[0].cpu().numpy() + 1) / 2).transpose(1, 2, 0).clip(0, 1)
        
        axes[i, 0].imshow(to_display(image))
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(to_display(masked_image.float()))
        axes[i, 1].set_title('Damaged')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(to_display(pred_image))
        axes[i, 2].set_title('EdgeConnect Restored')
        axes[i, 2].axis('off')
        
        axes[i, 3].imshow(mask[0, 0].cpu().float().numpy(), cmap='gray')
        axes[i, 3].set_title('Mask')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/edgeconnect_results.png', dpi=150)
    plt.show()

visualize_results(edge_gen, inpaint_gen, test_dataset)

In [13]:
# Download Outputs
import shutil
import os
from IPython.display import FileLink

# 1. Zip Checkpoints
if os.path.exists('/kaggle/working/checkpoints'):
    shutil.make_archive('checkpoints_download', 'zip', '/kaggle/working/checkpoints')

# 2. Zip Results
if os.path.exists('/kaggle/working/results'):
    shutil.make_archive('results_download', 'zip', '/kaggle/working/results')

print("⬇️ CLICK TO DOWNLOAD:")
display(FileLink('checkpoints_download.zip'))
display(FileLink('results_download.zip'))

⬇️ CLICK TO DOWNLOAD:


/kaggle/working/checkpoints_download.zip

/kaggle/working/results_download.zip